<div style="background:linear-gradient(135deg,#7A4E35 0%,#C99B72 100%);border-radius:14px;padding:22px 26px;color:#fff;font-family:Segoe UI,Arial,sans-serif;">
  <div style="font-size:13px;letter-spacing:1px;opacity:.9;text-transform:uppercase;">Generative AI Summer Bootcamp · Najran University</div>
  <div style="font-size:28px;font-weight:800;margin-top:5px;">RoomCraft AI — Interior Design Assistant</div>
  <div style="font-size:16px;opacity:.95;margin-top:5px;">Capstone Project by Ahad Dera · Multimodal Track</div>
</div>


> **Fixed version:** Open in a fresh Google Colab runtime, select **T4 GPU**, then choose **Runtime → Run all**.


# RoomCraft AI

**Student:** Ahad Dera  
**Track:** B — Multimodal Generative AI  

RoomCraft AI analyzes a room photo and creates practical interior-design recommendations for colors, furniture, lighting, curtains, layout, and décor. The user can choose a preferred style and budget, ask a visual question, and receive the answer in Arabic or English.

> This notebook is ready to run from top to bottom in Google Colab using a T4 GPU.


## Project objectives

- Analyze an uploaded room image using a vision-language model.
- Generate a clear image caption and answer visual questions.
- Use a generative language model to produce personalized interior-design advice.
- Provide a simple Gradio interface for a live demo.
- Prepare deployment files for GitHub and a Hugging Face Space.


## How to run

1. Choose **Runtime → Change runtime type → T4 GPU**.
2. Run all cells from top to bottom.
3. Upload a room photo in the Gradio interface.
4. Choose the design style, budget level, and output language.
5. Click **Design My Room**.

Only **Track B (Multimodal)** is used in this project.


<div style="border:1px solid #16304A33;border-left:6px solid #16304A;background:#EDF1F6;border-radius:10px;padding:14px 18px;margin:6px 0;font-family:Segoe UI,Arial,sans-serif;">
<div style="font-weight:800;color:#16304A;font-size:15px;margin-bottom:4px;">🔧 First: turn on the free GPU</div>
<div style="font-size:15px;color:#1B2A3A;line-height:1.5;">Click <b>Runtime → Change runtime type → T4 GPU → Save</b>. Everything also runs on CPU, just slower. Then run Part 0 below.</div></div>

## Part 0 — Shared setup (run this once)

Installs the libraries used by all three tracks and defines helpers you will reuse. This is the same toolkit you practised in Weeks 1–3.

In [ ]:
# One-time setup (~2-4 minutes). Safe to re-run on a fresh Colab runtime.
# Pillow is pinned and force-reinstalled to avoid mixed-version PIL import errors.
!pip -q install --force-reinstall --no-cache-dir "pillow==11.3.0"
!pip -q install -U "transformers>=4.56,<6" accelerate "gradio>=5" requests

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

print("Setup complete — run the next cells in order.")


In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", DEVICE)
if DEVICE == "cpu":
    print("Tip: enable the T4 GPU (see the setup box) for much faster generation.")

### Text-generation model

RoomCraft AI uses **Qwen2.5-1.5B-Instruct**, an open model that fits the free Colab T4 GPU and turns visual observations into structured interior-design recommendations.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
print("LLM loaded:", LLM_ID)


def chat(prompt, system=None, max_new_tokens=420, **gen_kwargs):
    """Send a prompt to Qwen and return the generated text."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=False,
    ).to(llm.device)
    out = llm.generate(
        inputs,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        **gen_kwargs,
    )
    return tokenizer.decode(
        out[0][inputs.shape[1]:], skip_special_tokens=True
    ).strip()

print(chat("Introduce RoomCraft AI in one short sentence.", do_sample=False))


<div style="border:1px solid #24783033;border-left:6px solid #247830;background:#E9F5EC;border-radius:10px;padding:14px 18px;margin:6px 0;font-family:Segoe UI,Arial,sans-serif;">
<div style="font-weight:800;color:#247830;font-size:15px;margin-bottom:4px;">💡 Decoding knobs you already know</div>
<div style="font-size:15px;color:#1B2A3A;line-height:1.5;">Pass <code style="background:#EEF3F8;padding:1px 5px;border-radius:4px;">do_sample=False</code> for factual answers, or <code style="background:#EEF3F8;padding:1px 5px;border-radius:4px;">do_sample=True, temperature=0.8, top_p=0.9</code> for creative ones — straight from the Week-2 lab.</div></div>

## Part 1 — Multimodal models

The application uses BLIP for image captioning and visual question answering. The generated visual observations are passed to Qwen to create personalized design recommendations.


In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration, BlipForQuestionAnswering
from PIL import Image
import requests
from io import BytesIO

CAPTION_ID = "Salesforce/blip-image-captioning-base"
VQA_ID = "Salesforce/blip-vqa-base"

caption_proc = BlipProcessor.from_pretrained(CAPTION_ID)
caption_model = BlipForConditionalGeneration.from_pretrained(CAPTION_ID).to(DEVICE)

vqa_proc = BlipProcessor.from_pretrained(VQA_ID)
vqa_model = BlipForQuestionAnswering.from_pretrained(VQA_ID).to(DEVICE)
print("Vision models loaded: BLIP captioning + BLIP VQA")


def load_image(path_or_url):
    """Load a PIL image from a local path or URL."""
    if str(path_or_url).startswith("http"):
        response = requests.get(path_or_url, timeout=30)
        response.raise_for_status()
        return Image.open(BytesIO(response.content)).convert("RGB")
    return Image.open(path_or_url).convert("RGB")


In [ ]:
def caption_image(image):
    """Return a concise English description of an image."""
    image = image.convert("RGB")
    inputs = caption_proc(image, return_tensors="pt").to(DEVICE)
    output = caption_model.generate(**inputs, max_new_tokens=45)
    return caption_proc.decode(output[0], skip_special_tokens=True)


def ask_image(image, question):
    """Answer a natural-language question about an image."""
    image = image.convert("RGB")
    inputs = vqa_proc(image, question, return_tensors="pt").to(DEVICE)
    output = vqa_model.generate(**inputs, max_new_tokens=30)
    return vqa_proc.decode(output[0], skip_special_tokens=True)


# Quick model test using a public interior image.
sample_url = "https://images.unsplash.com/photo-1616486338812-3dadae4b4ace?w=900"
img = load_image(sample_url)
print("Caption:", caption_image(img))
print("VQA:", ask_image(img, "What type of room is this?"))


### Uploading your own image

The final Gradio interface below includes an image uploader, so you do not need to edit image paths. During the demo, upload a clear photo of a living room, bedroom, hallway, or majlis.


### Why this is multimodal

RoomCraft AI combines image understanding and text generation:

1. **BLIP Captioning** recognizes the room and visible objects.
2. **BLIP VQA** answers a user question about the image.
3. **Qwen** converts those observations into organized design advice.


## Part 2 — RoomCraft AI design engine

The function below creates a personalized design report using the image caption, visual answer, preferred style, budget, and output language.


In [ ]:
PROJECT_NAME = "RoomCraft AI"
CREATOR = "Ahad Dera"

STYLE_GUIDES = {
    "Modern": "clean lines, practical furniture, neutral colors, and uncluttered styling",
    "Luxury": "elegant materials, refined lighting, layered textures, and a premium appearance",
    "Classic": "balanced furniture, timeless details, warm woods, and formal symmetry",
    "Minimal": "simple forms, fewer objects, functional storage, and calm colors",
    "Saudi Contemporary": "modern comfort with tasteful Saudi and Najrani-inspired details",
}

BUDGET_GUIDES = {
    "Low": "Prioritize paint, rearranging existing furniture, textiles, and affordable accessories.",
    "Medium": "Allow selected furniture replacement, improved lighting, curtains, and feature décor.",
    "High": "Allow custom furniture, premium finishes, architectural lighting, and full styling.",
}


def design_room(image, style, budget, question, language):
    """Analyze a room image and generate an interior-design report."""
    if image is None:
        return "Please upload a clear room image first. / الرجاء رفع صورة واضحة للغرفة أولًا."

    question = (question or "What is the room type and what are its main visible features?").strip()
    caption = caption_image(image)
    visual_answer = ask_image(image, question)

    output_rule = (
        "Write the complete answer in clear Arabic. Keep the project title in English."
        if language == "Arabic"
        else "Write the complete answer in clear professional English."
    )

    prompt = f"""
You are RoomCraft AI, a careful and practical interior-design assistant created by Ahad Dera.
Use only the visual observations below. Do not claim exact dimensions, materials, structural safety,
or hidden room features that cannot be confirmed from the image.

Image caption: {caption}
User's visual question: {question}
Visual answer: {visual_answer}
Preferred style: {style} — {STYLE_GUIDES[style]}
Budget: {budget} — {BUDGET_GUIDES[budget]}

Create a useful report with these exact headings:
1. Room Analysis
2. Color Palette
3. Furniture and Layout
4. Lighting
5. Curtains and Textiles
6. Decor Details
7. Three Priority Actions
8. Important Note

Under Important Note, state that recommendations are AI-generated and that structural, electrical,
and construction changes should be reviewed by a qualified professional.
{output_rule}
"""

    report = chat(
        prompt,
        system="You give tasteful, realistic, concise interior-design recommendations.",
        do_sample=True,
        temperature=0.65,
        top_p=0.9,
        repetition_penalty=1.08,
    )

    prefix = f"""**Project:** {PROJECT_NAME}  
**Created by:** {CREATOR}  

**Vision caption:** {caption}  
**Visual Q&A:** {visual_answer}  

"""
    return prefix + report

# Test the design engine on the sample image.
print(design_room(img, "Modern", "Medium", "What type of room is this?", "English")[:1200])


## Part 3 — Gradio live application

Run the next cell to open RoomCraft AI. Upload a room photo, select the desired style and budget, and generate the report.


### Demo inputs

- **Room image:** living room, bedroom, hallway, majlis, or home office.
- **Style:** Modern, Luxury, Classic, Minimal, or Saudi Contemporary.
- **Budget:** Low, Medium, or High.
- **Visual question:** an optional question such as “What type of room is this?”
- **Language:** Arabic or English.


In [ ]:
import gradio as gr

with gr.Blocks(title="RoomCraft AI") as demo:
    gr.Markdown(
        """
        # 🏠 RoomCraft AI
        ### Interior Design Assistant by Ahad Dera
        Upload a room photo and receive personalized recommendations for colors, furniture,
        lighting, curtains, layout, and décor.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label="Upload a room image")
            style_input = gr.Dropdown(
                choices=list(STYLE_GUIDES.keys()), value="Modern", label="Preferred style"
            )
            budget_input = gr.Radio(
                choices=list(BUDGET_GUIDES.keys()), value="Medium", label="Budget level"
            )
            question_input = gr.Textbox(
                value="What type of room is this and what are its main visible features?",
                label="Question about the image",
            )
            language_input = gr.Radio(
                choices=["Arabic", "English"], value="Arabic", label="Report language"
            )
            submit_button = gr.Button("Design My Room", variant="primary")

        with gr.Column(scale=1):
            output = gr.Markdown(label="RoomCraft AI Report")

    submit_button.click(
        fn=design_room,
        inputs=[image_input, style_input, budget_input, question_input, language_input],
        outputs=output,
    )

    gr.Markdown(
        "*AI-generated recommendations are for inspiration. Consult qualified professionals "
        "before structural, electrical, or construction changes.*"
    )

# Temporary public link for the live presentation.
demo.launch(share=True, debug=False)


### Track B interface

This project uses `gr.Blocks` because it needs several inputs: an image, style, budget, question, and language. The output is a formatted design report.


## Part 4 — Create deployment files

Run the next cell. It writes `app.py`, `requirements.txt`, `README.md`, and `PROJECT_WRITEUP.md` into the Colab Files panel. Download all four files and upload them to the GitHub repository.


### Deployment notes

The generated `app.py` contains the complete RoomCraft AI application. On Hugging Face Spaces, CPU loading can be slow. The notebook demo on Colab T4 is the recommended live presentation version.


In [ ]:
# Create all files needed for GitHub and optional Hugging Face deployment.
app_py = 'import gradio as gr\nimport torch\nfrom transformers import (\n    AutoModelForCausalLM,\n    AutoTokenizer,\n    BlipProcessor,\n    BlipForConditionalGeneration,\n    BlipForQuestionAnswering,\n)\n\nDEVICE = "cuda" if torch.cuda.is_available() else "cpu"\nLLM_ID = "Qwen/Qwen2.5-1.5B-Instruct"\n\n# Models are loaded once when the app starts.\ntokenizer = AutoTokenizer.from_pretrained(LLM_ID)\nllm = AutoModelForCausalLM.from_pretrained(\n    LLM_ID,\n    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,\n    device_map="auto",\n)\ncaption_proc = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")\ncaption_model = BlipForConditionalGeneration.from_pretrained(\n    "Salesforce/blip-image-captioning-base"\n).to(DEVICE)\nvqa_proc = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")\nvqa_model = BlipForQuestionAnswering.from_pretrained(\n    "Salesforce/blip-vqa-base"\n).to(DEVICE)\n\nSTYLE_GUIDES = {\n    "Modern": "clean lines, practical furniture, neutral colors, and uncluttered styling",\n    "Luxury": "elegant materials, refined lighting, layered textures, and a premium appearance",\n    "Classic": "balanced furniture, timeless details, warm woods, and formal symmetry",\n    "Minimal": "simple forms, fewer objects, functional storage, and calm colors",\n    "Saudi Contemporary": "modern comfort with tasteful Saudi and Najrani-inspired details",\n}\nBUDGET_GUIDES = {\n    "Low": "Prioritize paint, rearranging existing furniture, textiles, and affordable accessories.",\n    "Medium": "Allow selected furniture replacement, improved lighting, curtains, and feature decor.",\n    "High": "Allow custom furniture, premium finishes, architectural lighting, and full styling.",\n}\n\n\ndef chat(prompt, system=None, max_new_tokens=420, **gen_kwargs):\n    messages = []\n    if system:\n        messages.append({"role": "system", "content": system})\n    messages.append({"role": "user", "content": prompt})\n    inputs = tokenizer.apply_chat_template(\n        messages, add_generation_prompt=True, return_tensors="pt", return_dict=False\n    ).to(llm.device)\n    output = llm.generate(\n        inputs,\n        max_new_tokens=max_new_tokens,\n        pad_token_id=tokenizer.eos_token_id,\n        **gen_kwargs,\n    )\n    return tokenizer.decode(\n        output[0][inputs.shape[1]:], skip_special_tokens=True\n    ).strip()\n\n\ndef caption_image(image):\n    inputs = caption_proc(image.convert("RGB"), return_tensors="pt").to(DEVICE)\n    output = caption_model.generate(**inputs, max_new_tokens=45)\n    return caption_proc.decode(output[0], skip_special_tokens=True)\n\n\ndef ask_image(image, question):\n    inputs = vqa_proc(image.convert("RGB"), question, return_tensors="pt").to(DEVICE)\n    output = vqa_model.generate(**inputs, max_new_tokens=30)\n    return vqa_proc.decode(output[0], skip_special_tokens=True)\n\n\ndef design_room(image, style, budget, question, language):\n    if image is None:\n        return "Please upload a room image first. / الرجاء رفع صورة للغرفة أولًا."\n    question = (question or "What type of room is this?").strip()\n    caption = caption_image(image)\n    visual_answer = ask_image(image, question)\n    output_rule = (\n        "Write the complete answer in clear Arabic. Keep the project title in English."\n        if language == "Arabic"\n        else "Write the complete answer in clear professional English."\n    )\n    prompt = f"""\nYou are RoomCraft AI, a careful interior-design assistant created by Ahad Dera.\nUse only these visual observations. Do not invent dimensions or hidden features.\nImage caption: {caption}\nQuestion: {question}\nVisual answer: {visual_answer}\nStyle: {style} — {STYLE_GUIDES[style]}\nBudget: {budget} — {BUDGET_GUIDES[budget]}\nCreate a concise report with: Room Analysis, Color Palette, Furniture and Layout,\nLighting, Curtains and Textiles, Decor Details, Three Priority Actions, Important Note.\nThe note must say that structural/electrical work requires a qualified professional.\n{output_rule}\n"""\n    report = chat(\n        prompt,\n        system="Give tasteful, realistic and practical interior-design advice.",\n        do_sample=True,\n        temperature=0.65,\n        top_p=0.9,\n        repetition_penalty=1.08,\n    )\n    return (\n        f"**Project:** RoomCraft AI  \\n**Created by:** Ahad Dera  \\n"\n        f"**Vision caption:** {caption}  \\n**Visual Q&A:** {visual_answer}  \\n\\n{report}"\n    )\n\n\nwith gr.Blocks(title="RoomCraft AI") as demo:\n    gr.Markdown("# 🏠 RoomCraft AI\\n### Interior Design Assistant by Ahad Dera")\n    with gr.Row():\n        with gr.Column():\n            image_input = gr.Image(type="pil", label="Upload a room image")\n            style_input = gr.Dropdown(list(STYLE_GUIDES), value="Modern", label="Style")\n            budget_input = gr.Radio(list(BUDGET_GUIDES), value="Medium", label="Budget")\n            question_input = gr.Textbox(\n                value="What type of room is this and what are its main visible features?",\n                label="Question about the image",\n            )\n            language_input = gr.Radio(["Arabic", "English"], value="Arabic", label="Language")\n            button = gr.Button("Design My Room", variant="primary")\n        with gr.Column():\n            output = gr.Markdown()\n    button.click(\n        design_room,\n        [image_input, style_input, budget_input, question_input, language_input],\n        output,\n    )\n    gr.Markdown("*AI suggestions are for inspiration. Consult professionals before structural or electrical changes.*")\n\nif __name__ == "__main__":\n    demo.launch()\n'
requirements = """transformers>=4.56,<6
torch
accelerate
gradio>=5
pillow==11.3.0
requests
"""
readme = '# RoomCraft AI 🏠\n\n**RoomCraft AI** is a multimodal interior-design assistant created by **Ahad Dera** for the Najran University Generative AI Summer Bootcamp capstone.\n\n## What it does\n\nThe user uploads a room photo, chooses a preferred style and budget, and receives recommendations for:\n\n- Color palette\n- Furniture and layout\n- Lighting\n- Curtains and textiles\n- Décor details\n- Three priority actions\n\nThe application can produce the report in Arabic or English.\n\n## AI models\n\n- `Salesforce/blip-image-captioning-base` for image captioning\n- `Salesforce/blip-vqa-base` for visual question answering\n- `Qwen/Qwen2.5-1.5B-Instruct` for personalized design recommendations\n\n## Run in Google Colab\n\n1. Open `RoomCraft_AI_Capstone_Ahad_Dera.ipynb` in Google Colab.\n2. Enable a **T4 GPU**.\n3. Run all cells in order.\n4. Upload a room image in the Gradio interface.\n\n## Limitations and responsible use\n\nThe system may misunderstand visual details and does not measure dimensions or assess structural safety. Its suggestions are for inspiration only. Structural, electrical, and construction changes must be reviewed by qualified professionals.\n'
writeup = "# RoomCraft AI — One-Page Project Write-up\n\n**Student:** Ahad Dera  \n**Program:** Generative AI Summer Bootcamp, Najran University  \n**Track:** Multimodal Generative AI\n\n## Problem\n\nChoosing suitable colors, furniture, lighting, curtains, and décor for a room can be difficult for non-specialists. Professional consultations may not always be immediately available, and users often need a quick starting point before making design decisions.\n\n## Solution\n\nRoomCraft AI is a multimodal interior-design assistant. A user uploads a room photo, chooses a preferred design style and budget level, asks an optional question, and selects Arabic or English. The application analyzes the image and returns an organized design report with a room analysis, color palette, furniture and layout recommendations, lighting, curtains and textiles, décor details, and three priority actions.\n\n## Generative AI pipeline\n\nThe application combines three open models. BLIP Image Captioning creates a description of the image. BLIP Visual Question Answering answers the user's visual question. Qwen2.5-1.5B-Instruct receives these observations together with the selected style and budget, then generates personalized interior-design recommendations. A Gradio interface connects the full pipeline into a simple web application.\n\n## Value and uniqueness\n\nRoomCraft AI makes basic design guidance accessible to Arabic-speaking users and includes a Saudi Contemporary style option. The project connects my interest in interior design with practical generative AI skills learned during the bootcamp.\n\n## Limitations and responsible use\n\nThe model can misunderstand images and cannot confirm measurements, materials, electrical safety, or structural conditions. The recommendations are inspirational rather than professional plans. Users are clearly advised to consult qualified professionals before structural, electrical, or construction work.\n\n## Future improvements\n\nFuture versions could generate a matching color-palette image, support before-and-after visualization, recognize room dimensions, save design reports, and connect recommendations to locally available furniture products.\n"

files_to_create = {
    "app.py": app_py,
    "requirements.txt": requirements,
    "README.md": readme,
    "PROJECT_WRITEUP.md": writeup,
}

for filename, content in files_to_create.items():
    with open(filename, "w", encoding="utf-8") as file:
        file.write(content)
    print("Created:", filename)

print("\nDownload these four files from the Colab Files panel and upload them to GitHub.")


### Free-tier note

The Colab demo uses a T4 GPU and is best for the live presentation. A Hugging Face Space can be added if available, but the GitHub repository should always include the notebook, `app.py`, `requirements.txt`, `README.md`, and `PROJECT_WRITEUP.md`.


## Final checklist

- [x] Project title and student name added.
- [x] Multimodal Track B selected.
- [x] Image captioning implemented.
- [x] Visual question answering implemented.
- [x] Personalized generative design report implemented.
- [x] Arabic and English output supported.
- [x] Gradio live demo implemented.
- [x] Deployment files generated.
- [ ] Run all cells and test with at least two room photos.
- [ ] Download the completed notebook and generated files.
- [ ] Upload all files to the public GitHub repository.
- [ ] Paste the public GitHub repository URL into the assignment.
